In [ ]:
# -*- coding: utf-8 -*-
"""
这个脚本用于分析 "mix_stability" 实验的结果。
(重构版 v3: 针对 *每个 Token 步骤* 计算群体指标并保存)

它会：
1. 设置一个 "基准" 目录 (BATCH_SIZE=1)。
2. 遍历所有其他 BATCH_SIZE 目录 (2, 4, 8, 16)。
3. 对每个 (Question, BatchSize) 组合:
    a. 加载 "基准" (B=1, run_00) 的 logits。
    b. (关键) 同时加载 B={batch_size} 下的 *所有* (例如 10 次) 运行的 logits。
    c. 找到所有运行都与基准保持一致的 "最短稳定前缀" (num_steps_to_compare)。
    d. (核心) 遍历这个稳定前缀中的 *每一步 t*：
        i.  计算在该步 t 上的所有新指标 (sigma_agg, D_max, D_min, sigma_agg_b1, D_max_b1, D_min_b1)。
        ii. (!!) 将 (Q, BS, t, ...metrics) 作为 *新的一行* 添加到最终 CSV 列表中。
4. 将这个长格式的 CSV 文件保存。
"""

import os
import pickle
import torch
import torch.nn.functional as F
import numpy as np
import csv
from tqdm import tqdm
import sys

# --- 1. 配置 (必须与数据生成脚本一致) ---
NUM_RUNS = 10 
NUM_MMLU_QUESTIONS = 8 
NUM_EXTRA_QUESTIONS = 2
BATCH_SIZES_TO_COMPARE = [2, 4, 8, 16]
BASELINE_DIR = "mix_stability_reports_llama3.2_B1_fp16"
BASE_REPORT_DIR = "mix_stability_reports_llama3.2_B"
# (!!) 新的 CSV 文件名
CSV_REPORT_FILE = "stability_analysis_report_llama3.2_PER_TOKEN_METRICS_fp16.csv" 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用设备: {device}")


# --- 2. 辅助函数 ---

def load_run_data(path, device):
    """安全地加载 .pkl 文件并将其内容移动到指定设备。"""
    if not os.path.exists(path):
        tqdm.write(f"⚠️ 警告: 找不到文件 {path}，跳过...")
        return None, None
    try:
        with open(path, "rb") as f:
            data = pickle.load(f)
        tokens = data['tokens'].to('cpu') 
        logits = [l.to('cpu') for l in data['logits']]
        return tokens, logits
    except Exception as e:
        tqdm.write(f"❌ 错误: 加载 {path} 失败: {e}")
        return None, None

def find_divergence(tokens_run, tokens_baseline):
    """比较两个 token 序列，返回第一个分歧点的索引。-1 表示无分歧。"""
    compare_len = min(len(tokens_run), len(tokens_baseline))
    for i in range(compare_len):
        if tokens_run[i] != tokens_baseline[i]:
            return i # 在第 i 步分歧
    if len(tokens_run) != len(tokens_baseline):
        return compare_len # 序列长度不同，在末尾分歧
    return -1 # 完全一致

# --- 3. 确定要分析的问题列表 (简化版) ---

q_dirs_to_process = []
q_dirs_to_process.extend([f"question_{i:03d}" for i in range(NUM_MMLU_QUESTIONS)])
q_dirs_to_process.extend([f"extra_{i:03d}" for i in range(NUM_EXTRA_QUESTIONS)])
print(f"配置为分析 {NUM_MMLU_QUESTIONS} MMLU 问题和 {NUM_EXTRA_QUESTIONS} 额外问题。")
print(f"总计 {len(q_dirs_to_process)} 个问题目录将被分析。")


# --- 4. 主分析循环 ---

all_results_for_csv = [] 

if not os.path.exists(BASELINE_DIR):
    print(f"❌ 严重错误: 找不到基准目录 {BASELINE_DIR}！")
    sys.exit(1)

for batch_size in BATCH_SIZES_TO_COMPARE:
    print(f"\n===== 正在分析 BATCH_SIZE = {batch_size} (对比 {BASELINE_DIR}) =====")
    
    current_report_dir = f"{BASE_REPORT_DIR}{batch_size}_fp16"

    for q_dir_name in tqdm(q_dirs_to_process, desc=f"Analyzing B={batch_size}"):
        
        # --- 4.1 加载基准 (B=1, Run=00) ---
        baseline_path = os.path.join(BASELINE_DIR, q_dir_name, "run_00.pkl")
        baseline_tokens, baseline_logits = load_run_data(baseline_path, 'cpu')
        
        if baseline_tokens is None:
            tqdm.write(f"⚠️ 警告: 找不到基准 {baseline_path}，跳过问题 {q_dir_name}")
            continue

        # --- 4.2 加载 B={batch_size} 的 *所有* 运行 ---
        all_runs_logits = [] 
        all_runs_tokens = [] 
        
        for k in range(NUM_RUNS):
            run_filename = os.path.join(current_report_dir, q_dir_name, f"run_{k:02d}.pkl")
            run_tokens, run_logits = load_run_data(run_filename, 'cpu')
            if run_tokens is not None:
                all_runs_tokens.append(run_tokens)
                all_runs_logits.append(run_logits)
        
        if not all_runs_logits:
            tqdm.write(f"⚠️ 警告: {q_dir_name} B={batch_size} 没有任何有效运行数据。")
            continue
        
        # --- 4.3 找到 "群体最短稳定前缀" ---
        # (这对于确保比较的是同一步骤的 logits 至关重要)
        
        # 1. 找到每次运行与基准的分歧点
        divergence_points = [find_divergence(t, baseline_tokens) for t in all_runs_tokens]
        valid_div_points = [dp for dp in divergence_points if dp != -1]
        
        # 2. 找到 *最早* 的分歧点
        min_group_divergence_step = min(valid_div_points) if valid_div_points else float('inf')
        
        # 3. 找到 *最短* 的 logits 列表长度
        min_baseline_len = len(baseline_logits)
        min_runs_len = min(len(logits) for logits in all_runs_logits)
        
        # 4. 最终可以比较的步数
        num_steps_to_compare = int(min(min_group_divergence_step, min_baseline_len, min_runs_len))

        if num_steps_to_compare == 0:
            # 如果在第 0 步就分歧了，或没有 logits，我们无法计算任何 token 指标
            print('something weird')
            del baseline_tokens, baseline_logits, all_runs_tokens, all_runs_logits
            if device == "cuda": torch.cuda.empty_cache()
            continue

        # --- 4.4 (!!) 遍历每一步 t，计算指标并 *立即保存* ---
        
        try:
            for t in range(num_steps_to_compare):
                # 1. 获取基准的概率 (移至 GPU)
                probs_b1 = F.softmax(baseline_logits[t].to(device).float(), dim=-1)
                
                # 2. 获取 r 次运行的概率 (构建一个 (r, V) 的张量)
                probs_runs_t = torch.stack(
                    [F.softmax(run_logits[t].to(device).float(), dim=-1) for run_logits in all_runs_logits]
                ) # Shape: (r, V)
                
                r = probs_runs_t.shape[0] # r_valid_runs

                # --- 3b. "与基准 (B=1) 比较" 的指标 ---
                
                # D(i, b1) for all i
                d_b1_list = torch.sum(torch.abs(probs_runs_t - probs_b1), dim=-1) # Shape: (r)
                
                d_max_b1 = torch.max(d_b1_list).item()
                d_min_b1 = torch.min(d_b1_list).item()

                # sigma_agg-b1
                sq_diffs_b1 = (probs_runs_t - probs_b1)**2
                mean_var_vs_b1 = torch.mean(torch.mean(sq_diffs_b1, dim=0))
                sigma_agg_b1 = torch.sqrt(mean_var_vs_b1).item()
                
                # --- 3a. "组内" 稳定性指标 ---
                
                # sigma_agg
                probs_mean = torch.mean(probs_runs_t, dim=0)
                sq_diffs_internal = (probs_runs_t - probs_mean)**2
                mean_var_internal = torch.mean(torch.mean(sq_diffs_internal, dim=0))
                sigma_agg = torch.sqrt(mean_var_internal).item()

                # D_max / D_min (O(r^2) 计算)
                d_pairs = []
                d_max = 0.0
                d_min = 0.0
                if r > 1:
                    for i1 in range(r):
                        for i2 in range(i1 + 1, r):
                            d_val = torch.sum(torch.abs(probs_runs_t[i1] - probs_runs_t[i2])).item()
                            d_pairs.append(d_val)
                
                if d_pairs: # 确保列表不为空
                    d_max = max(d_pairs)
                    d_min = min(d_pairs)

                # --- 4. (!!) 立即将此 Token 步骤的结果添加到 CSV 列表 ---
                all_results_for_csv.append([
                    q_dir_name, 
                    batch_size, 
                    t,              # <-- Token 步骤索引
                    sigma_agg, 
                    d_max, 
                    d_min,
                    sigma_agg_b1, 
                    d_max_b1, 
                    d_min_b1
                ])

                # 清理这步的 GPU 内存
                del probs_b1, probs_runs_t, d_b1_list
                if device == "cuda": torch.cuda.empty_cache()

        except Exception as e:
            tqdm.write(f"❌ 错误: 在 {q_dir_name} B={batch_size} 的步骤 {t} 计算中失败: {e}")
            # 跳过这个 (q, bs) 组合的剩余步骤
            del baseline_tokens, baseline_logits, all_runs_tokens, all_runs_logits
            if device == "cuda": torch.cuda.empty_cache()
            continue

        # --- 4.5 (!!) 清理内存 (已移到循环末尾) ---
        del baseline_tokens, baseline_logits, all_runs_tokens, all_runs_logits
        if device == "cuda": torch.cuda.empty_cache()

# --- 5. (!!) 保存 CSV 报告 ---

if not all_results_for_csv:
    print("\n❌ 错误: 没有收集到任何结果。无法生成 CSV。")
    sys.exit(1)

print(f"\n✅ 分析完成。正在将 *每个 Token* 的群体指标保存到 {CSV_REPORT_FILE}...")

try:
    with open(CSV_REPORT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        # (!!) 写入新的、针对 Token 的表头
        writer.writerow([
            "Question", "BatchSize", "Token_Step",
            "Sigma_Agg", "D_Max", "D_Min",
            "Sigma_Agg_B1", "D_Max_B1", "D_Min_B1"
        ])
        writer.writerows(all_results_for_csv)
    
    print(f"🎉 成功保存报告！")

except Exception as e:
    print(f"❌ 错误: 保存 CSV 失败: {e}")

# --- 6. (!!) 删除了最终总结 ---
print("\n--- 脚本执行完毕 ---")

In [ ]:
#按照baseline的probability对每一个token分箱，然后计算每个token的10次运行误差的平均值
#存储相关数据等待后续输出
# -*- coding: utf-8 -*-
"""
这个脚本用于执行 "分箱平均误差" 分析。

它会：
1. 遍历每个模型 (gemma3, llama3.2, ...) 和每个 BatchSize (2, 4, 8, 16)。
2. 加载 B=1 (基准) 和 B={bs} (所有 r 次运行) 的 .pkl 数据。
3. 遍历每个稳定的 Token 步骤 t。
4. 在步骤 t，计算 V 个 (x, y) 数据点：
    - x = 基准概率 P_b1(v(j))
    - y = 平均绝对误差 AvgAbsError_j = (1/r) * sum_i |P_i(j) - P_b1(j)|
5. 将 V 个数据点按 x 轴（基准概率）分为 10 个分箱。
6. 计算每个分箱内 y 值的平均值。
7. 将结果 [Model, BS, t, Bin, Avg_Error] 保存到一个新的 CSV 文件中。
"""

import os
import pickle
import torch
import torch.nn.functional as F
import numpy as np
import csv
from tqdm import tqdm
import sys

# --- 1. 配置 ---

# (!!) 定义你的模型名称，这必须与文件夹名称匹配
# 例如 "mix_stability_reports_gemma3_B1" -> model_name = "gemma3"
MODEL_NAMES = [
    # "gemma3", 
    "llama3.2", 
    # "qwen3",
    # 'deepseek_qwen3'
] 
# # (!!) 从你的 CSV 列表中，模型名称似乎是:
# MODEL_NAMES = ["deepseek_qwen3", "gemma3", "llama3.2", "qwen3"]


BATCH_SIZES_TO_COMPARE = [2, 4, 8, 16]
BASELINE_BS = 1
NUM_RUNS = 10 
NUM_MMLU_QUESTIONS = 8 
NUM_EXTRA_QUESTIONS = 2

# (!!) 报告目录的模板
REPORT_DIR_TEMPLATE = "mix_stability_reports_{model}_B{bs}_fp16" # 例如 "mix_stability_reports_gemma3_B2"

# (!!) 定义分箱
NUM_BINS = 10
# np.linspace(0, 1, 11) 会创建 [0.0, 0.1, 0.2, ..., 1.0] (11 个边缘，定义 10 个箱)
BINS = np.linspace(0, 1, NUM_BINS + 1)
# 为 bin 创建标签 (例如 "[0.0-0.1)")
BIN_LABELS = [f"[{BINS[i]:.1f}-{BINS[i+1]:.1f})" for i in range(NUM_BINS)]
# 最后一个标签特殊处理 "[1.0]"
BIN_LABELS[-1] = f"[{BINS[-2]:.1f}-{BINS[-1]:.1f}]"

# (!!) 新的 CSV 文件名
CSV_REPORT_FILE = "stability_analysis_report_BINNED_METRICS_fp16.csv" 

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用设备: {device}")


# --- 2. 辅助函数 (与之前相同) ---

def load_run_data(path, device):
    """安全地加载 .pkl 文件并将其内容移动到 CPU。"""
    if not os.path.exists(path):
        tqdm.write(f"⚠️ 警告: 找不到文件 {path}，跳过...")
        return None, None
    try:
        with open(path, "rb") as f:
            data = pickle.load(f)
        tokens = data['tokens'].to('cpu') 
        logits = [l.to('cpu') for l in data['logits']]
        return tokens, logits
    except Exception as e:
        tqdm.write(f"❌ 错误: 加载 {path} 失败: {e}")
        return None, None

def find_divergence(tokens_run, tokens_baseline):
    compare_len = min(len(tokens_run), len(tokens_baseline))
    for i in range(compare_len):
        if tokens_run[i] != tokens_baseline[i]:
            return i
    if len(tokens_run) != len(tokens_baseline):
        return compare_len 
    return -1

# --- 3. 确定要分析的问题列表 (与之前相同) ---

q_dirs_to_process = []
q_dirs_to_process.extend([f"question_{i:03d}" for i in range(NUM_MMLU_QUESTIONS)])
q_dirs_to_process.extend([f"extra_{i:03d}" for i in range(NUM_EXTRA_QUESTIONS)])
print(f"配置为分析 {NUM_MMLU_QUESTIONS} MMLU 问题和 {NUM_EXTRA_QUESTIONS} 额外问题。")
print(f"将使用 {NUM_BINS} 个分箱: {BIN_LABELS}")

# --- 4. 主分析循环 (重构) ---

all_results_for_csv = [] 

for model_name in tqdm(MODEL_NAMES, desc="Processing Models"):
    
    baseline_dir_template = REPORT_DIR_TEMPLATE.format(model=model_name, bs=BASELINE_BS)
    if not os.path.exists(baseline_dir_template):
        print(f"⚠️ 警告: 找不到模型 {model_name} 的基准目录 {baseline_dir_template}，跳过...")
        continue

    for batch_size in tqdm(BATCH_SIZES_TO_COMPARE, desc=f"Model {model_name} BS", leave=False):
        
        current_report_dir_template = REPORT_DIR_TEMPLATE.format(model=model_name, bs=batch_size)
        if not os.path.exists(current_report_dir_template):
            print(f"⚠️ 警告: 找不到 {current_report_dir_template}，跳过 BATCH_SIZE={batch_size}")
            continue

        for q_dir_name in tqdm(q_dirs_to_process, desc=f"Analyzing B={batch_size}", leave=False):
            
            # --- 4.1 加载基准 (B=1, Run=00) ---
            baseline_path = os.path.join(baseline_dir_template, q_dir_name, "run_00.pkl")
            baseline_tokens, baseline_logits = load_run_data(baseline_path, 'cpu')
            
            if baseline_tokens is None:
                continue # 警告已在 load_run_data 中打印

            # --- 4.2 加载 B={batch_size} 的 *所有* 运行 ---
            all_runs_logits = [] 
            all_runs_tokens = [] 
            for k in range(NUM_RUNS):
                run_filename = os.path.join(current_report_dir_template, q_dir_name, f"run_{k:02d}.pkl")
                run_tokens, run_logits = load_run_data(run_filename, 'cpu')
                if run_tokens is not None:
                    all_runs_tokens.append(run_tokens)
                    all_runs_logits.append(run_logits)
            
            if not all_runs_logits:
                continue

            # --- 4.3 找到 "群体最短稳定前缀" ---
            divergence_points = [find_divergence(t, baseline_tokens) for t in all_runs_tokens]
            valid_div_points = [dp for dp in divergence_points if dp != -1]
            min_group_divergence_step = min(valid_div_points) if valid_div_points else float('inf')
            min_baseline_len = len(baseline_logits)
            min_runs_len = min(len(logits) for logits in all_runs_logits)
            num_steps_to_compare = int(min(min_group_divergence_step, min_baseline_len, min_runs_len))

            if num_steps_to_compare == 0:
                print('wrong23412')
                del baseline_tokens, baseline_logits, all_runs_tokens, all_runs_logits
                if device == "cuda": torch.cuda.empty_cache()
                continue

            # --- 4.4 (!!) 核心逻辑: 遍历步骤 t，执行分箱计算 ---
            try:
                for t in range(num_steps_to_compare):
                    
                    # 1. (X 轴) B=1 基准概率
                    #    (V,)
                    probs_b1 = F.softmax(baseline_logits[t].to(device).float(), dim=-1)
                    
                    # 2. r 次运行的概率
                    #    (r, V)
                    probs_runs = torch.stack(
                        [F.softmax(run_logits[t].to(device).float(), dim=-1) for run_logits in all_runs_logits]
                    )
                    
                    # 3. (Y 轴) 平均绝对误差
                    #    (V,)
                    avg_abs_error = torch.mean(torch.abs(probs_runs - probs_b1), dim=0)

                    # --- 4. 分箱 (在 CPU 上使用 numpy) ---
                    # (V,)
                    x_data_probs = probs_b1.cpu().numpy()
                    # (V,)
                    y_data_errors = avg_abs_error.cpu().numpy()

                    # np.digitize: 计算 V 个 token 中，每个 token 属于哪个分箱
                    # (V,)
                    # BINS = [0.0, 0.1, ..., 1.0]。
                    # 值为 0.05 的 x 会得到索引 1。值为 0.15 的 x 会得到索引 2。
                    # 值为 1.0 的 x 会得到索引 10。
                    bin_indices = np.digitize(x_data_probs, BINS, right=False)
                    
                    # 确保索引在 [1, 10] 范围内 (np.digitize 默认 0->0, 1.1->11)
                    bin_indices = np.clip(bin_indices, 1, NUM_BINS)


                    # --- 5. 聚合每个分箱的平均 Y 值 ---
                    for bin_id in range(1, NUM_BINS + 1): # 遍历 1 到 10
                        # 找到所有属于这个 bin_id 的 token
                        mask = (bin_indices == bin_id)
                        
                        if np.any(mask):
                            # 计算这个分箱的平均误差
                            avg_error_in_bin = np.mean(y_data_errors[mask])
                        else:
                            # print('wrong422')
                            # 这个分箱里没有 token
                            avg_error_in_bin = 0.0
                        
                        bin_label = BIN_LABELS[bin_id - 1] # 索引从 0 开始

                        # --- 6. 保存结果 ---
                        all_results_for_csv.append([
                            model_name,
                            q_dir_name,
                            batch_size,
                            t,            # Token 步骤
                            bin_label,    # 分箱标签 (X 轴)
                            avg_error_in_bin  # 柱高 (Y 轴)
                        ])

                    # 清理这步的 GPU 内存
                    del probs_b1, probs_runs, avg_abs_error
                    if device == "cuda": torch.cuda.empty_cache()

            except Exception as e:
                tqdm.write(f"❌ 错误: 在 {model_name} {q_dir_name} B={batch_size} 的步骤 {t} 计算中失败: {e}")
                del baseline_tokens, baseline_logits, all_runs_tokens, all_runs_logits
                if device == "cuda": torch.cuda.empty_cache()
                continue
            
            # 清理这个问题的内存
            del baseline_tokens, baseline_logits, all_runs_tokens, all_runs_logits
            if device == "cuda": torch.cuda.empty_cache()

# --- 5. 保存最终的 CSV 报告 ---

if not all_results_for_csv:
    print("\n❌ 错误: 没有收集到任何结果。无法生成 CSV。")
    sys.exit(1)

print(f"\n✅ 分析完成。正在将 *分箱后* 的指标保存到 {CSV_REPORT_FILE}...")

try:
    with open(CSV_REPORT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "Model", "Question", "BatchSize", "Token_Step",
            "Prob_Bin", "Binned_Avg_Error"
        ])
        writer.writerows(all_results_for_csv)
    
    print(f"🎉 成功保存报告！")

except Exception as e:
    print(f"❌ 错误: 保存 CSV 失败: {e}")

print("\n--- 脚本执行完毕 ---")
print(f"现在你可以使用 {CSV_REPORT_FILE} 来绘制你的分箱柱状图了。")

In [ ]:
# -*- coding: utf-8 -*-
# 按照logprob>-5的个数分箱
"""
这个脚本用于执行 "基于不确定性状态的分箱分析"。

它会：
1. 遍历模型、BatchSize 和步骤 t。
2. 在步骤 t，分析 B=1 基准模型的输出分布：
   - 计算有多少个 Token 的 log_prob > -5 (设为 count)。
   - 这个 count 代表了模型在这一步的"纠结程度"。
3. 将当前步骤 t 分配到一个 "Count Bin" (例如 "Count=1", "Count=2", "Count=3+").
4. 计算这一步的"总稳定性误差"：Step_Total_Error = sum(|P_runs - P_b1|)。
5. 保存结果 [Model, BS, t, Count_Bin, Step_Total_Error] 到 CSV。
"""

import os
import pickle
import torch
import torch.nn.functional as F
import numpy as np
import csv
from tqdm import tqdm
import sys

# --- 1. 配置 ---

MODEL_NAMES = [
    "gemma3",
    "llama3.2",
    # "qwen3",
    # 'deepseek_qwen3'
]

BATCH_SIZES_TO_COMPARE = [2, 4, 8, 16]
BASELINE_BS = 1
NUM_RUNS = 10
NUM_MMLU_QUESTIONS = 8
NUM_EXTRA_QUESTIONS = 2

REPORT_DIR_TEMPLATE = "mix_stability_reports_{model}_B{bs}"

# (!!) 新的配置：LogProb 阈值和分箱定义
LOGPROB_THRESHOLD = -5.0

CSV_REPORT_FILE = "stability_logprob_report_UNCERTAINTY_BINS.csv"

device = "cuda"
print(f"使用设备: {device}")
print(f"LogProb 阈值设置为: {LOGPROB_THRESHOLD}")

# --- 2. 辅助函数 (保持不变) ---
# ... (此处省略 load_run_data 和 find_divergence，与你之前的代码完全相同) ...
def load_run_data(path, device):
    if not os.path.exists(path):
        tqdm.write(f"⚠️ 警告: 找不到文件 {path}，跳过...")
        return None, None
    try:
        with open(path, "rb") as f:
            data = pickle.load(f)
        tokens = data['tokens'].to('cpu')
        logits = [l.to('cpu') for l in data['logits']]
        return tokens, logits
    except Exception as e:
        tqdm.write(f"❌ 错误: 加载 {path} 失败: {e}")
        return None, None

def find_divergence(tokens_run, tokens_baseline):
    compare_len = min(len(tokens_run), len(tokens_baseline))
    for i in range(compare_len):
        if tokens_run[i] != tokens_baseline[i]:
            return i
    if len(tokens_run) != len(tokens_baseline):
        return compare_len
    return -1

# --- 3. 问题列表 (保持不变) ---
q_dirs_to_process = []
q_dirs_to_process.extend([f"question_{i:03d}" for i in range(NUM_MMLU_QUESTIONS)])
q_dirs_to_process.extend([f"extra_{i:03d}" for i in range(NUM_EXTRA_QUESTIONS)])

# --- 4. 主分析循环 ---

all_results_for_csv = []

for model_name in tqdm(MODEL_NAMES, desc="Processing Models"):
    baseline_dir_template = REPORT_DIR_TEMPLATE.format(model=model_name, bs=BASELINE_BS)
    if not os.path.exists(baseline_dir_template):
        print(f"⚠️ 警告: 找不到基准目录 {baseline_dir_template}")
        continue

    for batch_size in tqdm(BATCH_SIZES_TO_COMPARE, desc=f"{model_name} BS"):
        current_dir = REPORT_DIR_TEMPLATE.format(model=model_name, bs=batch_size)
        if not os.path.exists(current_dir): 
            print('!!!!!wrong')
            continue

        for q_dir_name in tqdm(q_dirs_to_process, desc=f"BS={batch_size}"):
            # --- 4.1 & 4.2 加载数据 (保持不变) ---
            baseline_path = os.path.join(baseline_dir_template, q_dir_name, "run_00.pkl")
            b_tokens, b_logits = load_run_data(baseline_path, 'cpu')
            if b_tokens is None: 
                print('wrong35235235')
                continue

            runs_logits, runs_tokens = [], []
            for k in range(NUM_RUNS):
                rp = os.path.join(current_dir, q_dir_name, f"run_{k:02d}.pkl")
                rt, rl = load_run_data(rp, 'cpu')
                if rt is not None:
                    runs_tokens.append(rt)
                    runs_logits.append(rl)
            if not runs_logits: 
                print('error13214')
                continue

            # --- 4.3 确定稳定前缀 (保持不变) ---
            divs = [find_divergence(t, b_tokens) for t in runs_tokens]
            valid_divs = [d for d in divs if d != -1]
            min_div = min(valid_divs)
            limit = int(min(min_div, len(b_logits), min(len(rl) for rl in runs_logits)))

            if limit == 0: 
                print('error41341')
                continue

            # --- 4.4 (!!) 新的核心循环 ---
            try:
                for t in range(limit):
                    # --- A. 分析基准模型在这一步的"确定性" ---
                    # 将 logits 移到 GPU 计算以加速
                    logits_b1_t = b_logits[t].to(device).float()
                    # 计算 log_softmax 以获取 logprobs
                    logprobs_b1 = F.log_softmax(logits_b1_t, dim=-1)

                    # (关键) 计算有多少个 token 的 logprob > -5
                    count_gt_threshold = (logprobs_b1 > LOGPROB_THRESHOLD).sum().item()
                    # 获取对应的分箱标签 (例如 "Count=02")
                    bin_label = count_gt_threshold

                    # --- B. 计算这一步的整体误差 ---
                    probs_b1 = F.softmax(logits_b1_t, dim=-1)

                    # 堆叠所有 run 的这一步概率 (NUM_RUNS, V)
                    probs_runs_t = torch.stack([F.softmax(rl[t].to(device).float(), dim=-1) for rl in runs_logits])
                    

                    # 计算平均绝对误差向量 (V,)
                    avg_abs_error_vec = torch.mean(torch.abs(probs_runs_t - probs_b1), dim=0)
                    

                    
                    # (关键) 将向量求和，得到这一步的"总误差"标量
                    # 这个值越大，说明这一步的整体分布受 BatchSize 影响越严重
                    step_total_error = avg_abs_error_vec.sum().item()

                    # --- C. 记录结果 ---
                    # 注意：现在每一行代表一个"步骤 t"，而不是之前的"步骤 t 中的一个概率区间"
                    all_results_for_csv.append([
                        model_name,
                        q_dir_name,
                        batch_size,
                        t,              # Token Step
                        bin_label,      # 新的 Bin (基于 Count)
                        step_total_error # 这一步的总误差
                    ])

                    # 清理显存
                    del logits_b1_t, logprobs_b1, probs_b1, probs_runs_t, avg_abs_error_vec
                    if device == "cuda": torch.cuda.empty_cache()

            except Exception as e:
                tqdm.write(f"❌ Error at {model_name} {q_dir_name} t={t}: {e}")
                continue

            # 清理问题级内存
            del b_tokens, b_logits, runs_tokens, runs_logits
            if device == "cuda": torch.cuda.empty_cache()

# --- 5. 保存 CSV ---
if not all_results_for_csv:
    print("❌ 没有数据可保存。")
    sys.exit(1)

print(f"\n✅ 分析完成，正在保存到 {CSV_REPORT_FILE}...")
with open(CSV_REPORT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Model", "Question", "BatchSize", "Token_Step", "Count_Bin", "Step_Total_Error"])
    writer.writerows(all_results_for_csv)
print("🎉 报告已保存。")

In [ ]:
# -*- coding: utf-8 -*-
# 分析目标：
# 严格按照 Overleaf 定义，计算独立 Token 的波动性。
# - Standard Deviation (sigma_j), 使用 1/r (unbiased=False)
# - Range (R_j), 使用 (max - min)
"""
这个脚本用于执行 "基于独立Token的波动性分析"。

它会：
1. 遍历模型、BatchSize 和步骤 t。
2. 在步骤 t，分析 B=1 基准模型的输出分布：
   - 找出概率最高的 Top-K (例如 Top-10) 的 Token ID (v(j))。
3. 对于这 Top-K 个 Token 中的 *每一个 Token v(j)*：
   - 查看 B=N 的所有 R (例如 10) 次运行 (Runs)。
   - 提取 R 次运行中 *这一个特定 Token* 的概率 P(v(j))_i，i=1..r。
   - (!!) 严格按照数学公式计算:
     a) 标准差 sigma_j (分母为 r)
     b) 范围 R_j (max - min)
4. 保存结果，每一行代表一个 Top-K Token 在一个步骤 t 上的表现：
   [Model, BS, t, Rank, Token_ID, Prob_B1, Mean_Prob_Runs, Std_Prob_Runs (sigma_j), Range_Prob_Runs (R_j)]
"""

import os
import pickle
import torch
import torch.nn.functional as F
import numpy as np
import csv
from tqdm import tqdm
import sys

# --- 1. 配置 ---

MODEL_NAMES = [
    "gemma3",
    "llama3.2",
    # "qwen3",
    # 'deepseek_qwen3'
]

BATCH_SIZES_TO_COMPARE = [2, 4, 8, 16]
BASELINE_BS = 1
NUM_RUNS = 10
NUM_MMLU_QUESTIONS = 8
NUM_EXTRA_QUESTIONS = 2

REPORT_DIR_TEMPLATE = "mix_stability_reports_{model}_B{bs}"

# 要分析的 Top-K Token 数量
TOP_K_TO_ANALYZE = 10

CSV_REPORT_FILE = "stability_token_level_report_STD_RANGE.csv"

device = "cuda"
print(f"使用设备: {device}")
print(f"分析 Top-K Token: {TOP_K_TO_ANALYZE}")
print(f"NUM_RUNS (r) = {NUM_RUNS}")

# --- 2. 辅助函数 (保持不变) ---
def load_run_data(path, device):
    if not os.path.exists(path):
        tqdm.write(f"⚠️ 警告: 找不到文件 {path}，跳过...")
        return None, None
    try:
        with open(path, "rb") as f:
            data = pickle.load(f)
        tokens = data['tokens'].to('cpu')
        logits = [l.to('cpu') for l in data['logits']]
        return tokens, logits
    except Exception as e:
        tqdm.write(f"❌ 错误: 加载 {path} 失败: {e}")
        return None, None

def find_divergence(tokens_run, tokens_baseline):
    compare_len = min(len(tokens_run), len(tokens_baseline))
    for i in range(compare_len):
        if tokens_run[i] != tokens_baseline[i]:
            return i
    if len(tokens_run) != len(tokens_baseline):
        return compare_len
    return -1

# --- 3. 问题列表 (保持不变) ---
q_dirs_to_process = []
q_dirs_to_process.extend([f"question_{i:03d}" for i in range(NUM_MMLU_QUESTIONS)])
q_dirs_to_process.extend([f"extra_{i:03d}" for i in range(NUM_EXTRA_QUESTIONS)])

# --- 4. 主分析循环 ---

all_results_for_csv = []

for model_name in tqdm(MODEL_NAMES, desc="Processing Models"):
    baseline_dir_template = REPORT_DIR_TEMPLATE.format(model=model_name, bs=BASELINE_BS)
    if not os.path.exists(baseline_dir_template):
        print(f"⚠️ 警告: 找不到基准目录 {baseline_dir_template}")
        continue

    for batch_size in tqdm(BATCH_SIZES_TO_COMPARE, desc=f"{model_name} BS"):
        current_dir = REPORT_DIR_TEMPLATE.format(model=model_name, bs=batch_size)
        if not os.path.exists(current_dir): 
            print('!!!!!wrong')
            continue

        for q_dir_name in tqdm(q_dirs_to_process, desc=f"BS={batch_size}"):
            # --- 4.1 & 4.2 加载数据 (保持不变) ---
            baseline_path = os.path.join(baseline_dir_template, q_dir_name, "run_00.pkl")
            b_tokens, b_logits = load_run_data(baseline_path, 'cpu')
            if b_tokens is None: 
                print('wrong35235235')
                continue

            runs_logits, runs_tokens = [], []
            # 确保我们加载了 NUM_RUNS (r) 次运行
            for k in range(NUM_RUNS):
                rp = os.path.join(current_dir, q_dir_name, f"run_{k:02d}.pkl")
                rt, rl = load_run_data(rp, 'cpu')
                if rt is not None:
                    runs_tokens.append(rt)
                    runs_logits.append(rl)
            
            # 检查加载的运行次数是否与配置的 r 匹配
            if len(runs_logits) != NUM_RUNS:
                tqdm.write(f"⚠️ 警告: {q_dir_name} 期望 {NUM_RUNS} 次运行, 实际找到 {len(runs_logits)}。跳过...")
                continue

            # --- 4.3 确定稳定前缀 (保持不变) ---
            divs = [find_divergence(t, b_tokens) for t in runs_tokens]
            valid_divs = [d for d in divs if d != -1]
            min_div = min(valid_divs)
            limit = int(min(min_div, len(b_logits), min(len(rl) for rl in runs_logits)))

            if limit == 0: 
                print('error41341')
                continue

            # --- 4.4 核心循环 (独立 Token 分析) ---
            try:
                for t in range(limit):
                    # --- A. 获取 B=1 的概率和 Top-K Token ---
                    logits_b1_t = b_logits[t].to(device).float()
                    probs_b1 = F.softmax(logits_b1_t, dim=-1)
                    
                    top_k_probs_b1, top_k_indices_b1 = torch.topk(probs_b1, TOP_K_TO_ANALYZE)

                    # --- B. 获取 B=N (所有 Runs) 的概率 ---
                    # 堆叠所有 r 次运行的概率 (r, V)
                    probs_runs_t_stacked = torch.stack(
                        [F.softmax(rl[t].to(device).float(), dim=-1) for rl in runs_logits]
                    )

                    # --- C. 遍历 B=1 的 Top-K Token v(j) ---
                    for i in range(TOP_K_TO_ANALYZE):
                        token_rank = i + 1
                        token_id_j = top_k_indices_b1[i].item() # v(j)
                        prob_b1 = top_k_probs_b1[i].item()

                        # 提取 r 次运行中，这一个 token_id_j 的所有概率 P(v(j))_i
                        # 得到一个大小为 [r] (即 NUM_RUNS) 的 1D 张量
                        probs_for_this_token_all_runs = probs_runs_t_stacked[:, token_id_j]

                        # --- (!!) D. 严格按照您的公式计算统计数据 ---
                        
                        # 计算平均值 P-bar(v(j))
                        mean_prob_runs = probs_for_this_token_all_runs.mean().item()

                        # (!!) (关键修正) 计算 sigma_j (分母为 r)
                        # 我们设置 unbiased=False 来使用 1/r (总体标准差)
                        # 而不是 1/(r-1) (样本标准差)
                        std_prob_runs = probs_for_this_token_all_runs.std(unbiased=False).item()
                        
                        # 计算 R_j (Range)
                        max_prob_runs = probs_for_this_token_all_runs.max().item()
                        min_prob_runs = probs_for_this_token_all_runs.min().item()
                        range_prob_runs = max_prob_runs - min_prob_runs

                        # --- E. 记录这个 Token 的结果 ---
                        all_results_for_csv.append([
                            model_name,
                            q_dir_name,
                            batch_size,
                            t,                  # Token Step
                            token_rank,         # Rank (1-10)
                            token_id_j,         # Token ID v(j)
                            prob_b1,            # B=1 时的概率
                            mean_prob_runs,     # B=N 时的平均概率 (P-bar)
                            std_prob_runs,      # B=N 时的标准差 (sigma_j)
                            range_prob_runs     # B=N 时的范围 (R_j)
                        ])

                    # 清理显存 (在 t 循环内部)
                    del logits_b1_t, probs_b1, top_k_probs_b1, top_k_indices_b1
                    del probs_runs_t_stacked, probs_for_this_token_all_runs
                    if device == "cuda": torch.cuda.empty_cache()

            except Exception as e:
                tqdm.write(f"❌ Error at {model_name} {q_dir_name} t={t}: {e}")
                continue

            # 清理问题级内存
            del b_tokens, b_logits, runs_tokens, runs_logits
            if device == "cuda": torch.cuda.empty_cache()

# --- 5. 保存 CSV ---
if not all_results_for_csv:
    print("❌ 没有数据可保存。")
    sys.exit(1)

print(f"\n✅ 分析完成，正在保存到 {CSV_REPORT_FILE}...")
with open(CSV_REPORT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    # CSV 标头与您的定义保持一致
    writer.writerow([
        "Model", "Question", "BatchSize", "Token_Step", 
        "Token_Rank", "Token_ID", "Prob_B1", 
        "Mean_Prob_Runs", "Std_Prob_Runs (sigma_j)", "Range_Prob_Runs (R_j)"
    ])
    writer.writerows(all_results_for_csv)
print("🎉 报告已保存。")

In [ ]:
import pandas as pd

# 读取两个 CSV 文件
df1 = pd.read_csv('stability_token_level_report_STD_RANGE_1.csv')
df2 = pd.read_csv('stability_token_level_report_STD_RANGE_2.csv')

# 直接拼接（按行堆叠）
df_merged = pd.concat([df1, df2], ignore_index=True)

# 保存到新文件
df_merged.to_csv('stability_token_level_report_STD_RANGE.csv', index=False)


In [ ]:
# -*- coding: utf-8 -*-
# 分析目标：
# 严格按照 Overleaf 定义，计算独立 Token 的波动性。
# - Standard Deviation (sigma_j), 使用 1/r (unbiased=False)
# - Range (R_j), 使用 (max - min)
"""
这个脚本用于执行 "基于独立Token的波动性分析"。

它会：
1. 遍历模型、BatchSize 和步骤 t。
2. 在步骤 t，分析 B=1 基准模型的输出分布：
   - 找出概率最高的 Top-K (例如 Top-10) 的 Token ID (v(j))。
3. 对于这 Top-K 个 Token 中的 *每一个 Token v(j)*：
   - 查看 B=N 的所有 R (例如 10) 次运行 (Runs)。
   - 提取 R 次运行中 *这一个特定 Token* 的概率 P(v(j))_i，i=1..r。
   - (!!) 严格按照数学公式计算:
     a) 标准差 sigma_j (分母为 r)
     b) 范围 R_j (max - min)
4. 保存结果，每一行代表一个 Top-K Token 在一个步骤 t 上的表现：
   [Model, BS, t, Rank, Token_ID, Prob_B1, Mean_Prob_Runs, Std_Prob_Runs (sigma_j), Range_Prob_Runs (R_j)]
"""

import os
import pickle
import torch
import torch.nn.functional as F
import numpy as np
import csv
from tqdm import tqdm
import sys

# --- 1. 配置 ---

MODEL_NAMES = [
    "gemma3_270M",
    "gemma3_1B",
    "gemma3_4B",
    "gemma3",
    # "llama3.2",
    # "qwen3",
    # 'deepseek_qwen3'
]

BATCH_SIZES_TO_COMPARE = [2, 4, 8, 16]
BASELINE_BS = 1
NUM_RUNS = 10
NUM_MMLU_QUESTIONS = 8
NUM_EXTRA_QUESTIONS = 2

REPORT_DIR_TEMPLATE = "mix_stability_reports_{model}_B{bs}"

# 要分析的 Top-K Token 数量
TOP_K_TO_ANALYZE = 10

CSV_REPORT_FILE = "stability_token_level_report_STD_RANGE_gemma.csv"

device = "cuda"
print(f"使用设备: {device}")
print(f"分析 Top-K Token: {TOP_K_TO_ANALYZE}")
print(f"NUM_RUNS (r) = {NUM_RUNS}")

# --- 2. 辅助函数 (保持不变) ---
def load_run_data(path, device):
    if not os.path.exists(path):
        tqdm.write(f"⚠️ 警告: 找不到文件 {path}，跳过...")
        return None, None
    try:
        with open(path, "rb") as f:
            data = pickle.load(f)
        tokens = data['tokens'].to('cpu')
        logits = [l.to('cpu') for l in data['logits']]
        return tokens, logits
    except Exception as e:
        tqdm.write(f"❌ 错误: 加载 {path} 失败: {e}")
        return None, None

def find_divergence(tokens_run, tokens_baseline):
    compare_len = min(len(tokens_run), len(tokens_baseline))
    for i in range(compare_len):
        if tokens_run[i] != tokens_baseline[i]:
            return i
    if len(tokens_run) != len(tokens_baseline):
        return compare_len
    return -1

# --- 3. 问题列表 (保持不变) ---
q_dirs_to_process = []
q_dirs_to_process.extend([f"question_{i:03d}" for i in range(NUM_MMLU_QUESTIONS)])
q_dirs_to_process.extend([f"extra_{i:03d}" for i in range(NUM_EXTRA_QUESTIONS)])

# --- 4. 主分析循环 ---

all_results_for_csv = []

for model_name in tqdm(MODEL_NAMES, desc="Processing Models"):
    baseline_dir_template = REPORT_DIR_TEMPLATE.format(model=model_name, bs=BASELINE_BS)
    if not os.path.exists(baseline_dir_template):
        print(f"⚠️ 警告: 找不到基准目录 {baseline_dir_template}")
        continue

    for batch_size in tqdm(BATCH_SIZES_TO_COMPARE, desc=f"{model_name} BS"):
        current_dir = REPORT_DIR_TEMPLATE.format(model=model_name, bs=batch_size)
        if not os.path.exists(current_dir): 
            print('!!!!!wrong')
            continue

        for q_dir_name in tqdm(q_dirs_to_process, desc=f"BS={batch_size}"):
            # --- 4.1 & 4.2 加载数据 (保持不变) ---
            baseline_path = os.path.join(baseline_dir_template, q_dir_name, "run_00.pkl")
            b_tokens, b_logits = load_run_data(baseline_path, 'cpu')
            if b_tokens is None: 
                print('wrong35235235')
                continue

            runs_logits, runs_tokens = [], []
            # 确保我们加载了 NUM_RUNS (r) 次运行
            for k in range(NUM_RUNS):
                rp = os.path.join(current_dir, q_dir_name, f"run_{k:02d}.pkl")
                rt, rl = load_run_data(rp, 'cpu')
                if rt is not None:
                    runs_tokens.append(rt)
                    runs_logits.append(rl)
            
            # 检查加载的运行次数是否与配置的 r 匹配
            if len(runs_logits) != NUM_RUNS:
                tqdm.write(f"⚠️ 警告: {q_dir_name} 期望 {NUM_RUNS} 次运行, 实际找到 {len(runs_logits)}。跳过...")
                continue

            # --- 4.3 确定稳定前缀 (保持不变) ---
            divs = [find_divergence(t, b_tokens) for t in runs_tokens]
            valid_divs = [d for d in divs if d != -1]
            min_div = min(valid_divs)
            limit = int(min(min_div, len(b_logits), min(len(rl) for rl in runs_logits)))

            if limit == 0: 
                print('error41341')
                continue

            # --- 4.4 核心循环 (独立 Token 分析) ---
            try:
                for t in range(limit):
                    # --- A. 获取 B=1 的概率和 Top-K Token ---
                    logits_b1_t = b_logits[t].to(device).float()
                    probs_b1 = F.softmax(logits_b1_t, dim=-1)
                    
                    top_k_probs_b1, top_k_indices_b1 = torch.topk(probs_b1, TOP_K_TO_ANALYZE)

                    # --- B. 获取 B=N (所有 Runs) 的概率 ---
                    # 堆叠所有 r 次运行的概率 (r, V)
                    probs_runs_t_stacked = torch.stack(
                        [F.softmax(rl[t].to(device).float(), dim=-1) for rl in runs_logits]
                    )

                    # --- C. 遍历 B=1 的 Top-K Token v(j) ---
                    for i in range(TOP_K_TO_ANALYZE):
                        token_rank = i + 1
                        token_id_j = top_k_indices_b1[i].item() # v(j)
                        prob_b1 = top_k_probs_b1[i].item()

                        # 提取 r 次运行中，这一个 token_id_j 的所有概率 P(v(j))_i
                        # 得到一个大小为 [r] (即 NUM_RUNS) 的 1D 张量
                        probs_for_this_token_all_runs = probs_runs_t_stacked[:, token_id_j]

                        # --- (!!) D. 严格按照您的公式计算统计数据 ---
                        
                        # 计算平均值 P-bar(v(j))
                        mean_prob_runs = probs_for_this_token_all_runs.mean().item()

                        # (!!) (关键修正) 计算 sigma_j (分母为 r)
                        # 我们设置 unbiased=False 来使用 1/r (总体标准差)
                        # 而不是 1/(r-1) (样本标准差)
                        std_prob_runs = probs_for_this_token_all_runs.std(unbiased=False).item()
                        
                        # 计算 R_j (Range)
                        max_prob_runs = probs_for_this_token_all_runs.max().item()
                        min_prob_runs = probs_for_this_token_all_runs.min().item()
                        range_prob_runs = max_prob_runs - min_prob_runs

                        # --- E. 记录这个 Token 的结果 ---
                        all_results_for_csv.append([
                            model_name,
                            q_dir_name,
                            batch_size,
                            t,                  # Token Step
                            token_rank,         # Rank (1-10)
                            token_id_j,         # Token ID v(j)
                            prob_b1,            # B=1 时的概率
                            mean_prob_runs,     # B=N 时的平均概率 (P-bar)
                            std_prob_runs,      # B=N 时的标准差 (sigma_j)
                            range_prob_runs     # B=N 时的范围 (R_j)
                        ])

                    # 清理显存 (在 t 循环内部)
                    del logits_b1_t, probs_b1, top_k_probs_b1, top_k_indices_b1
                    del probs_runs_t_stacked, probs_for_this_token_all_runs
                    if device == "cuda": torch.cuda.empty_cache()

            except Exception as e:
                tqdm.write(f"❌ Error at {model_name} {q_dir_name} t={t}: {e}")
                continue

            # 清理问题级内存
            del b_tokens, b_logits, runs_tokens, runs_logits
            if device == "cuda": torch.cuda.empty_cache()

# --- 5. 保存 CSV ---
if not all_results_for_csv:
    print("❌ 没有数据可保存。")
    sys.exit(1)

print(f"\n✅ 分析完成，正在保存到 {CSV_REPORT_FILE}...")
with open(CSV_REPORT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    # CSV 标头与您的定义保持一致
    writer.writerow([
        "Model", "Question", "BatchSize", "Token_Step", 
        "Token_Rank", "Token_ID", "Prob_B1", 
        "Mean_Prob_Runs", "Std_Prob_Runs (sigma_j)", "Range_Prob_Runs (R_j)"
    ])
    writer.writerows(all_results_for_csv)
print("🎉 报告已保存。")

In [ ]:
# -*- coding: utf-8 -*-
# 分析目标：
# 严格按照 Overleaf 定义，计算独立 Token 的波动性。
# - Standard Deviation (sigma_j), 使用 1/r (unbiased=False)
# - Range (R_j), 使用 (max - min)
"""
这个脚本用于执行 "基于独立Token的波动性分析"。

它会：
1. 遍历模型、BatchSize 和步骤 t。
2. 在步骤 t，分析 B=1 基准模型的输出分布：
   - 找出概率最高的 Top-K (例如 Top-10) 的 Token ID (v(j))。
3. 对于这 Top-K 个 Token 中的 *每一个 Token v(j)*：
   - 查看 B=N 的所有 R (例如 10) 次运行 (Runs)。
   - 提取 R 次运行中 *这一个特定 Token* 的概率 P(v(j))_i，i=1..r。
   - (!!) 严格按照数学公式计算:
     a) 标准差 sigma_j (分母为 r)
     b) 范围 R_j (max - min)
4. 保存结果，每一行代表一个 Top-K Token 在一个步骤 t 上的表现：
   [Model, BS, t, Rank, Token_ID, Prob_B1, Mean_Prob_Runs, Std_Prob_Runs (sigma_j), Range_Prob_Runs (R_j)]
"""

import os
import pickle
import torch
import torch.nn.functional as F
import numpy as np
import csv
from tqdm import tqdm
import sys

# --- 1. 配置 ---

MODEL_NAMES = [
    "gemma3",
    "llama3.2",
    "qwen3",
    'deepseek_qwen3'
]

BATCH_SIZES_TO_COMPARE = [2, 4, 8, 16]
BASELINE_BS = 1
NUM_RUNS = 10
NUM_MMLU_QUESTIONS = 8
NUM_EXTRA_QUESTIONS = 2

REPORT_DIR_TEMPLATE = "tmp_mix_stability_reports_{model}_B{bs}"

# 要分析的 Top-K Token 数量
TOP_K_TO_ANALYZE = 10

CSV_REPORT_FILE = "tmp_stability_token_level_report_STD_RANGE.csv"

device = "cuda"
print(f"使用设备: {device}")
print(f"分析 Top-K Token: {TOP_K_TO_ANALYZE}")
print(f"NUM_RUNS (r) = {NUM_RUNS}")

# --- 2. 辅助函数 (保持不变) ---
def sparse_to_dense_logits(sparse_tensor, device, fill_value=float('-inf')):

    shape = sparse_tensor.shape
    indices = sparse_tensor._indices().to(device)
    values = sparse_tensor._values().to(device)
    
    dense_logits = torch.full(shape, fill_value, device=device, dtype=values.dtype)
    
    if indices.dim() == 2:
        indices = indices[0]
        
    dense_logits.scatter_(0, indices, values)
    
    return dense_logits
    
def load_run_data(path, device):
    if not os.path.exists(path):
        tqdm.write(f"⚠️ 警告: 找不到文件 {path}，跳过...")
        return None, None
    try:
        with open(path, "rb") as f:
            data = pickle.load(f)
        tokens = data['tokens'].to('cpu')
        logits = [l.to('cpu') for l in data['logits']]
        return tokens, logits
    except Exception as e:
        tqdm.write(f"❌ 错误: 加载 {path} 失败: {e}")
        return None, None

def find_divergence(tokens_run, tokens_baseline):
    compare_len = min(len(tokens_run), len(tokens_baseline))
    for i in range(compare_len):
        if tokens_run[i] != tokens_baseline[i]:
            return i
    if len(tokens_run) != len(tokens_baseline):
        return compare_len
    return -1

# --- 3. 问题列表 (保持不变) ---
q_dirs_to_process = []
q_dirs_to_process.extend([f"question_{i:03d}" for i in range(NUM_MMLU_QUESTIONS)])
q_dirs_to_process.extend([f"extra_{i:03d}" for i in range(NUM_EXTRA_QUESTIONS)])

# --- 4. 主分析循环 ---

all_results_for_csv = []

for model_name in tqdm(MODEL_NAMES, desc="Processing Models"):
    baseline_dir_template = REPORT_DIR_TEMPLATE.format(model=model_name, bs=BASELINE_BS)
    if not os.path.exists(baseline_dir_template):
        print(f"⚠️ 警告: 找不到基准目录 {baseline_dir_template}")
        continue

    for batch_size in tqdm(BATCH_SIZES_TO_COMPARE, desc=f"{model_name} BS"):
        current_dir = REPORT_DIR_TEMPLATE.format(model=model_name, bs=batch_size)
        if not os.path.exists(current_dir): 
            print('!!!!!wrong')
            continue

        for q_dir_name in tqdm(q_dirs_to_process, desc=f"BS={batch_size}"):
            # --- 4.1 & 4.2 加载数据 (保持不变) ---
            baseline_path = os.path.join(baseline_dir_template, q_dir_name, "run_00.pkl")
            b_tokens, b_logits = load_run_data(baseline_path, 'cpu')
            if b_tokens is None: 
                print('wrong35235235')
                continue

            runs_logits, runs_tokens = [], []
            # 确保我们加载了 NUM_RUNS (r) 次运行
            for k in range(NUM_RUNS):
                rp = os.path.join(current_dir, q_dir_name, f"run_{k:02d}.pkl")
                rt, rl = load_run_data(rp, 'cpu')
                if rt is not None:
                    runs_tokens.append(rt)
                    runs_logits.append(rl)
            
            # 检查加载的运行次数是否与配置的 r 匹配
            if len(runs_logits) != NUM_RUNS:
                tqdm.write(f"⚠️ 警告: {q_dir_name} 期望 {NUM_RUNS} 次运行, 实际找到 {len(runs_logits)}。跳过...")
                continue

            # --- 4.3 确定稳定前缀 (保持不变) ---
            divs = [find_divergence(t, b_tokens) for t in runs_tokens]
            valid_divs = [d for d in divs if d != -1]
            min_div = min(valid_divs)
            limit = int(min(min_div, len(b_logits), min(len(rl) for rl in runs_logits)))

            if limit == 0: 
                print('error41341')
                continue

            # --- 4.4 核心循环 (独立 Token 分析) ---
            try:
                for t in range(limit):
                    # --- A. 获取 B=1 的概率和 Top-K Token ---
                    sparse_b1 = b_logits[t].to(device)
                    logits_b1_dense = sparse_to_dense_logits(sparse_b1, device)
                    logits_b1_t = b_logits[t].to(device).float()
                    probs_b1 = F.softmax(logits_b1_dense.float(), dim=-1)
                    
                    top_k_probs_b1, top_k_indices_b1 = torch.topk(probs_b1, TOP_K_TO_ANALYZE)

                    # --- B. 获取 B=N (所有 Runs) 的概率 ---
                    # 堆叠所有 r 次运行的概率 (r, V)
                    batch_probs_list = []
                    for rl in runs_logits:
                        # 对每一次 Run 都进行同样的 稀疏 -> 稠密(-inf) -> Softmax 转换
                        sparse_run = rl[t].to(device)
                        dense_run = sparse_to_dense_logits(sparse_run, device)
                        prob_run = F.softmax(dense_run.float(), dim=-1)
                        batch_probs_list.append(prob_run)

                    # 堆叠所有 r 次运行的概率 (r, V)
                    probs_runs_t_stacked = torch.stack(batch_probs_list)
                    # probs_runs_t_stacked = torch.stack(
                    #     [F.softmax(rl[t].to(device).float(), dim=-1) for rl in runs_logits]
                    # )

                    # --- C. 遍历 B=1 的 Top-K Token v(j) ---
                    for i in range(TOP_K_TO_ANALYZE):
                        token_rank = i + 1
                        token_id_j = top_k_indices_b1[i].item() # v(j)
                        prob_b1 = top_k_probs_b1[i].item()

                        # 提取 r 次运行中，这一个 token_id_j 的所有概率 P(v(j))_i
                        # 得到一个大小为 [r] (即 NUM_RUNS) 的 1D 张量
                        probs_for_this_token_all_runs = probs_runs_t_stacked[:, token_id_j]

                        # --- (!!) D. 严格按照您的公式计算统计数据 ---
                        
                        # 计算平均值 P-bar(v(j))
                        mean_prob_runs = probs_for_this_token_all_runs.mean().item()

                        # (!!) (关键修正) 计算 sigma_j (分母为 r)
                        # 我们设置 unbiased=False 来使用 1/r (总体标准差)
                        # 而不是 1/(r-1) (样本标准差)
                        std_prob_runs = probs_for_this_token_all_runs.std(unbiased=False).item()
                        
                        # 计算 R_j (Range)
                        max_prob_runs = probs_for_this_token_all_runs.max().item()
                        min_prob_runs = probs_for_this_token_all_runs.min().item()
                        range_prob_runs = max_prob_runs - min_prob_runs

                        # --- E. 记录这个 Token 的结果 ---
                        all_results_for_csv.append([
                            model_name,
                            q_dir_name,
                            batch_size,
                            t,                  # Token Step
                            token_rank,         # Rank (1-10)
                            token_id_j,         # Token ID v(j)
                            prob_b1,            # B=1 时的概率
                            mean_prob_runs,     # B=N 时的平均概率 (P-bar)
                            std_prob_runs,      # B=N 时的标准差 (sigma_j)
                            range_prob_runs     # B=N 时的范围 (R_j)
                        ])

                    # 清理显存 (在 t 循环内部)
                    del logits_b1_t, probs_b1, top_k_probs_b1, top_k_indices_b1
                    del probs_runs_t_stacked, probs_for_this_token_all_runs
                    if device == "cuda": torch.cuda.empty_cache()

            except Exception as e:
                tqdm.write(f"❌ Error at {model_name} {q_dir_name} t={t}: {e}")
                continue

            # 清理问题级内存
            del b_tokens, b_logits, runs_tokens, runs_logits
            if device == "cuda": torch.cuda.empty_cache()

# --- 5. 保存 CSV ---
if not all_results_for_csv:
    print("❌ 没有数据可保存。")
    sys.exit(1)

print(f"\n✅ 分析完成，正在保存到 {CSV_REPORT_FILE}...")
with open(CSV_REPORT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    # CSV 标头与您的定义保持一致
    writer.writerow([
        "Model", "Question", "BatchSize", "Token_Step", 
        "Token_Rank", "Token_ID", "Prob_B1", 
        "Mean_Prob_Runs", "Std_Prob_Runs (sigma_j)", "Range_Prob_Runs (R_j)"
    ])
    writer.writerows(all_results_for_csv)
print("🎉 报告已保存。")